# Working with PMC XML: A Practical, Step-by-Step Guide

This notebook explains **how PMC full-text XML is structured**, how to **inspect it safely**, and how to **extract the text you actually want**. Everything here is based on *real PMC XML*, not theory.

---

## 1. Big Picture: What a PMC XML File Represents

A PMC XML file represents **scientific articles as a tree**.

At the highest level:

```
pmc-articleset
└── article
    ├── front
    ├── body
    └── back
```

* `pmc-articleset` → container (can hold one or more articles)
* `article` → the actual paper
* `front` → metadata + abstract
* `body` → full scientific text (THIS is what you want)
* `back` → references and appendices

---

## 2. Parsing XML into a Tree

```python
import xml.etree.ElementTree as ET
root = ET.fromstring(r.text)
```

### What this does

* Converts raw XML text into a **tree structure**
* `root` is now the **top node** of the XML

```python
print(root.tag)
```

Typical output:

```
pmc-articleset
```

---

## 3. Understanding the Root (`pmc-articleset`)

PMC wraps articles in a **set**, even if there is only one article.

```python
for child in root:
    print(child.tag)
```

Output:

```
article
```

➡️ The real paper is inside `root[0]`.

```python
article = root[0]
print(article.tag)
```

---

## 4. Inside `<article>`

```python
for child in article:
    print(child.tag)
```

Output:

```
front
body
back
```

These are the **three major sections** of every PMC article.

---

## 5. `front`: Metadata and Abstract

### What lives in `front`

| Tag             | Meaning          |
| --------------- | ---------------- |
| `article-title` | Paper title      |
| `contrib-group` | Authors          |
| `aff`           | Affiliations     |
| `abstract`      | Abstract text    |
| `kwd-group`     | Keywords         |
| `pub-date`      | Publication date |

### Example

```python
front = article.find("front")
print(front.find(".//article-title").text)
```

Abstract:

```python
for p in front.findall(".//abstract//p"):
    print(p.text)
```

---

## 6. `body`: Full PMC Text (MOST IMPORTANT)

The **actual scientific content** lives here.

```python
body = article.find("body")
```

### Structure

```
body
└── sec
    ├── title
    └── p
```

| Tag     | Meaning                               |
| ------- | ------------------------------------- |
| `sec`   | Section (Introduction, Methods, etc.) |
| `title` | Section heading                       |
| `p`     | Paragraph                             |

---

## 7. Understanding `<sec>`, `<title>`, and `<p>`

### Example XML

```xml
<sec>
  <title>Introduction</title>
  <p>First paragraph...</p>
  <p>Second paragraph...</p>
</sec>
```

### Meaning

* `title` → **section name**
* `p` → **paragraph text**

➡️ A section has **one title** and **many paragraphs**.

---

## 8. Inspecting Sections Manually (Recommended First Step)

```python
for sec in body:
    print("SECTION:", sec.find("title").text)
```

This lets you **see the structure before extracting text**.

---

## 9. Extracting PMC Text (Correct Way)

### Section-aware extraction (recommended)

```python
full_text = []

for sec in body.findall("sec"):
    title = sec.find("title")
    if title is not None:
        full_text.append("\n" + title.text.upper())
    for p in sec.findall("p"):
        if p.text:
            full_text.append(p.text)

text = "\n".join(full_text)
```

---

### Fast extraction (no structure)

```python
text = " ".join(p.text for p in article.findall(".//p") if p.text)
```

---

## 10. Cleaning Paragraph Text (Important Detail)

Paragraphs may contain inline tags:

```xml
<p>Liver transplantation <italic>remains</italic> the standard.</p>
```

To extract clean text:

```python
def clean_text(el):
    text = el.text or ""
    for child in el:
        text += (child.text or "") + (child.tail or "")
    return text.strip()
```

---

## 11. `back`: References and Appendices

### What lives in `back`

| Tag         | Meaning          |
| ----------- | ---------------- |
| `ref-list`  | References       |
| `ack`       | Acknowledgements |
| `fn-group`  | Footnotes        |
| `app-group` | Appendices       |

Example:

```python
back = article.find("back")
print(len(back.findall(".//ref")))
```

➡️ Usually **excluded** from NLP pipelines.

---

## 12. What to Use (and What to Ignore)

| Section | Use for NLP? | Reason                   |
| ------- | ------------ | ------------------------ |
| `front` | ⚠️ Partial   | metadata + abstract only |
| `body`  | ✅ YES        | real scientific text     |
| `back`  | ❌ No         | references pollute text  |

---

## 13. Safe XML Exploration Pattern (Always Use This)

```python
def inspect(el, level=0):
    print("  " * level + el.tag)
    for child in el:
        inspect(child, level + 1)

inspect(article)
```

This prints the **entire tree structure**.

---

## 14. Final Mental Model (Remember This)

> A PMC paper is a **tree of sections**.
> Each section has a **title** and **paragraphs**.
> The real text lives in `article → body → sec → p`.

---

## 15. What This Enables Next

Once you understand this, you can:

* Build a **full-text corpus**
* Chunk text for **RAG / LLMs**
* Filter by **license and OA**
* Store text with **section-level metadata**



## PMC XML Navigation: Functionality Reference Table

| Function / Concept | Example | What It Does | When to Use |
|-------------------|--------|--------------|-------------|
| `find(tag)` | `article.find("body")` | Returns the **first matching direct child** element | When you know the tag is **one level down** |
| `findall(tag)` | `body.findall("sec")` | Returns **all matching direct child** elements | Iterating over sections, paragraphs |
| `find(".//tag")` | `article.find(".//article-title")` | Finds the **first occurrence anywhere below** | Deeply nested tags |
| `findall(".//tag")` | `article.findall(".//p")` | Finds **all occurrences anywhere below** | Debugging, quick text checks |
| `"tag"` | `sec.find("title")` | Searches **only direct children** | Structure-aware extraction |
| `".//tag"` | `front.findall(".//abstract//p")` | Recursive search across subtree | Abstracts, metadata |
| `.text` | `p.text` | Text **before first child tag** | Never use alone for paragraphs |
| `.tail` | `child.tail` | Text **after a child tag** | Needed to avoid missing words |
| Iterate children | `for child in el:` | Walks through child elements | Inspecting structure |
| Check for `None` | `if el is not None:` | Prevents runtime errors | Always required |
| Tree walk | `inspect(el)` | Prints full XML hierarchy | Learning / debugging |
| Structure-aware path | `body → sec → p` | Preserves section semantics | NLP, RAG, corpora |
| Recursive shortcut | `.//p` | Ignores structure | Fast but noisy |
| Exclude `back` | `article.find("back")` | Avoid references | Clean text pipelines |

---

### Golden Rule

> **Structure-aware extraction (`body → sec → p`) beats convenience (`.//p`) every time.**

## XML Functionalities Cheat Sheet (with 5 Examples Each)

The examples below use **simple generic XML** so the behavior is easy to understand and applies everywhere.

---

### Sample XML used in examples

```xml
<root>
  <book>
    <title>XML Guide</title>
    <author>Jane</author>
    <chapter>
      <p>Hello <b>World</b>!</p>
    </chapter>
  </book>
  <book>
    <title>Another Book</title>
  </book>
</root>

| Functionality            | Example 1                            | Example 2                       | Example 3                 | Example 4                | Example 5                     |
| ------------------------ | ------------------------------------ | ------------------------------- | ------------------------- | ------------------------ | ----------------------------- |
| `find(tag)`              | `root.find("book")` → first `<book>` | `book.find("title")`            | `book.find("author")`     | `book.find("chapter")`   | `book.find("missing") → None` |
| `findall(tag)`           | `root.findall("book")`               | `len(root.findall("book")) = 2` | `book.findall("title")`   | `book.findall("author")` | `book.findall("x") → []`      |
| `find(".//tag")`         | `root.find(".//title")`              | `root.find(".//p")`             | `root.find(".//b")`       | `book.find(".//p")`      | `root.find(".//x") → None`    |
| `findall(".//tag")`      | `root.findall(".//title")`           | `root.findall(".//p")`          | `root.findall(".//book")` | `book.findall(".//b")`   | `root.findall(".//x") → []`   |
| Direct child search      | `book.find("title")`                 | `book.find("chapter")`          | `chapter.find("p")`       | `root.find("book")`      | `book.find("p") → None`       |
| Recursive search (`.//`) | `book.find(".//p")`                  | `root.find(".//author")`        | `root.find(".//b")`       | `chapter.find(".//b")`   | `root.find(".//title")`       |
| `.text`                  | `p.text → "Hello "`                  | `title.text`                    | `author.text`             | `b.text → "World"`       | `.text misses punctuation`    |
| `.tail`                  | `b.tail → "!"`                       | Combine text + tail             | `.tail belongs to child`  | `.tail may be None`      | Needed for full sentence      |
| Iterating children       | `for c in root:`                     | `for c in book:`                | `list(book)`              | `book[0].tag`            | Child order preserved         |
| Defensive `None` check   | `el = find()`                        | `if el is None:`                | `el.text if el else ""`   | Avoids crashes           | Required in real XML          |
| Structure-aware path     | `book → chapter → p`                 | Keeps meaning                   | Preserves hierarchy       | Clean extraction         | Best for NLP                  |
| Recursive shortcut       | `.//p`                               | Fast                            | Noisy                     | Ignores structure        | Debugging only                |

| XML Form  | Python String | Rendered Text | Meaning / Why It Exists                        |
| --------- | ------------- | ------------- | ---------------------------------------------- |
| `&lt;`    | `<`           | `<`           | Less-than sign (cannot appear raw in XML text) |
| `&gt;`    | `>`           | `>`           | Greater-than sign                              |
| `&amp;`   | `&`           | `&`           | Ampersand                                      |
| `&quot;`  | `"`           | `"`           | Double quote                                   |
| `&apos;`  | `'`           | `'`           | Apostrophe                                     |
| `&#160;`  | `\xa0`        | (space)       | **Non-breaking space** (`&nbsp;`)              |
| `&#8209;` | `\u2011`      | -             | Non-breaking hyphen                            |
| `&#8211;` | `\u2013`      | –             | En dash                                        |
| `&#8212;` | `\u2014`      | —             | Em dash                                        |
| `&#8220;` | `\u201c`      | “             | Left double quote                              |
| `&#8221;` | `\u201d`      | ”             | Right double quote                             |
| `&#8216;` | `\u2018`      | ‘             | Left single quote                              |
| `&#8217;` | `\u2019`      | ’             | Right single quote                             |
| `&#8230;` | `\u2026`      | …             | Ellipsis                                       |
| `&#176;`  | `°`           | °             | Degree symbol                                  |
| `&#177;`  | `±`           | ±             | Plus–minus                                     |
| `&#8804;` | `≤`           | ≤             | Less-than or equal                             |
| `&#8805;` | `≥`           | ≥             | Greater-than or equal                          |
| `&#956;`  | `μ`           | μ             | Greek mu (micro)                               |
| `&#945;`  | `α`           | α             | Greek alpha                                    |
| `&#946;`  | `β`           | β             | Greek beta                                     |
| `&#947;`  | `γ`           | γ             | Greek gamma                                    |
| `&#916;`  | `Δ`           | Δ             | Capital delta                                  |
| `&#8722;` | `−`           | −             | True minus sign (not ASCII `-`)                |
| `&#215;`  | `×`           | ×             | Multiplication sign                            |


# PMC XML Tag Reference Guide

This document lists the most common tags found in PubMed Central (PMC) articles using the JATS/NLM DTD format.

### **Article Metadata & Structure**
| Tag Name | Meaning | Description |
| :--- | :--- | :--- |
| `<front>` | **Front Matter** | Contains all metadata (journal info, title, authors, abstract). |
| `<body>` | **Article Body** | Contains the narrative content (Intro, Methods, Results, etc.). |
| `<back>` | **Back Matter** | Contains references, acknowledgments, and appendices. |
| `<article-title>` | **Article Title** | The main title of the paper. |
| `<journal-title>` | **Journal Name** | The name of the publishing journal. |
| `<abstract>` | **Abstract** | High-level summary of the article. |
| `<sec>` | **Section** | A logical division of the text (e.g., "Introduction"). Can be nested. |
| `<title>` | **Title** | The heading/name of a section, table, or figure. |
| `<p>` | **Paragraph** | The standard block for narrative text. |
| `<notes>` | **Notes** | General notes or additional information provided by authors. |
| `<ack>` | **Acknowledgments** | A section dedicated to thanking people or organizations. |
| `<glossary>` | **Glossary** | A list of terms and their definitions. |

### **Citations & References**
| Tag Name | Meaning | Description |
| :--- | :--- | :--- |
| `<xref>` | **Cross-Reference** | A link to a bibliography entry, figure, or table (e.g., "[1]"). |
| `<ref-list>` | **Reference List** | The list of all cited sources at the end of the paper. |
| `<ext-link>` | **External Link** | A URL to a website or external database. |
| `<contrib>` | **Contributor** | Information about an author or researcher. |
| `<aff>` | **Affiliation** | The institution or hospital an author belongs to. |
| `<author-notes>` | **Author Notes** | Footnotes or specific notes related to the authors. |
| `<corresp>` | **Correspondence** | Information regarding the corresponding author. |
| `<collab>` | **Collaboration** | Names of groups or consortia that contributed to the work. |

### **Formatting & Informatics**
| Tag Name | Meaning | Description |
| :--- | :--- | :--- |
| `<italic>`, `<i>` | **Italic** | Italicized text for emphasis or scientific names. |
| `<bold>`, `<b>` | **Bold** | Bolded text for headers or emphasis. |
| `<sup>` | **Superscript** | Raised text (often used for powers or footnote links). |
| `<sub>` | **Subscript** | Lowered text (often used in chemical formulas like H₂O). |
| `<list>` | **List** | A container for bulleted or numbered items. |
| `<list-item>` | **List Item** | An individual entry within a `<list>`. |
| `<def-list>` | **Definition List** | A list of terms and their definitions (Term/Def pairs). |
| `<term>` | **Term** | The word or phrase being defined in a `<def-list>`. |
| `<def>` | **Definition** | The explanation for a `<term>`. |
| `<fn>` | **Footnote** | A specific note, often linked from the text using a superscript. |
| `<underline>` | **Underline** | Underlined text (less common than italics). |

### **Data & Assets**
| Tag Name | Meaning | Description |
| :--- | :--- | :--- |
| `<table>` | **Table** | The standard HTML-like container for data tables. |
| `<tr>` / `<td>` | **Row / Cell** | Standard table row and data cell tags. |
| `<table-wrap>` | **Table Wrapper** | Container for a table along with its title and caption. |
| `<table-wrap-foot>` | **Table Footer** | Notes or legends appearing at the bottom of a table. |
| `<thead>` / `<tbody>` | **Table Head/Body** | Structural divisions of a table. |
| `<th>` | **Table Header Cell** | A cell specifically designated as a column or row header. |
| `<caption>` | **Caption** | The descriptive text for a table, figure, or media object. |
| `<fig>` | **Figure** | Container for an image, chart, or diagram. |
| `<graphic>` | **Graphic File** | Reference to the actual image file (JPEG/TIFF). |
| `<inline-graphic>` | **Inline Graphic** | A small graphic embedded directly in the text line. |
| `<media>` | **Media** | External media files like videos, audio, or animations. |
| `<supplementary-material>` | **Supplements** | Files or data provided alongside the main article. |
| `<disp-formula>` | **Display Formula** | A mathematical equation shown on its own line. |
| `<mml:math>` | **MathML** | Mathematical formulas encoded for software reading. |

### **Chemical & Special Content**
| Tag Name | Meaning | Description |
| :--- | :--- | :--- |
| `<chem-struct>` | **Chemical Structure** | Specific encoding for chemical formulas. |
| `<kwd>` | **Keyword** | Indexing words for the article. |
| `<sc>` | **Small Caps** | Text displayed in small capital letters. |

### **Publication & Identification**
| Tag Name | Meaning | Description |
| :--- | :--- | :--- |
| `<article-id>` | **Article ID** | Electronic identifiers (DOI, PMID, PMCID). |
| `<pub-date>` | **Publication Date** | When the article was published (online or print). |
| `<volume>` | **Volume** | The volume number of the journal. |
| `<issue>` | **Issue** | The issue number within a volume. |
| `<elocation-id>` | **E-Location ID** | Electronic page identifier for digital-only journals. |
| `<permissions>` | **Permissions** | Copyright and licensing information (e.g., Creative Commons). |

### **Funding & Finance**
| Tag Name | Meaning | Description |
| :--- | :--- | :--- |
| `<funding-group>` | **Funding Group** | Container for all financial support information. |
| `<award-group>` | **Award Group** | Details of a specific grant or award. |
| `<award-id>` | **Award ID** | The grant number or identifier. |
| `<funding-source>` | **Funding Source** | The name of the agency providing the funds. |
